# Problemas de Decisão Markovianos (MDP)


Este *notebook* Jupyter funciona como material de apoio para a compreensão dos Problemas de Decisão de Markov (MDP). Utilizámos as implementações no módulo `mdp.py`.

Este notebook inclui também um breve resumo de dois dos principais algoritmos para a resolução destes problemas.

Vamos importar tudo do módulo mdp para começar.

In [6]:
from mdp import *
from notebook import psource, pseudocode, plot_pomdp_utility

## ÍNDICE

* Visão geral
* MDP
* Grid MDP
* Iteração de valor
    - Visualização de iteração de valor
* Iteração de política
* Apendice

## VISÃO GERAL

Antes de começarmos a brincar com as implementações reais, vamos rever algumas coisas sobre MDPs.

- Um processo estocástico tem a **propriedade de Markov** se a distribuição de probabilidade condicional dos estados futuros do processo (condicional aos estados passados ​​e presentes) depende apenas do estado presente, não da sequência de eventos que o precederam. Fonte: [Wikipedia](https://pt.wikipedia.org/wiki/Propriedade_de_Markov)

É possível modelar muitos fenómenos diferentes como processos de Markov aproveitando a frexibilidade da definição de estado.


- Os MDP ajudam-nos a lidar com ambientes totalmente observáveis ​​e não determinísticos/estocásticos. Para lidar com casos parcialmente observáveis ​​e estocásticos, utilizamos a generalização de MDPs denominados POMDPs (processo de decisão de Markov parcialmente observável).

O nosso objetivo geral para resolver um MDP é criar uma política que nos oriente para selecionar a melhor ação em cada estado, de modo a maximizar a soma esperada das recompensas futuras.

## MDP

Para começar, vamos dar uma vista de olhos à implementação da classe MDP definida em mdp.py.

A docstring diz-nos tudo o que é necessário para definir um MDP, nomeadamente:

- conjunto de estados
- ações
- estado inicial
- modelo de transição
- uma função de recompensa

Cada um deles é implementado como métodos:

In [7]:
%psource MDP

class MDP:
    """A Markov Decision Process, defined by an initial state, transition model,
    and reward function. We also keep track of a gamma value, for use by
    algorithms. The transition model is represented somewhat differently from
    the text. Instead of P(s' | s, a) being a probability number for each
    state/state/action triplet, we instead have T(s, a) return a
    list of (p, s') pairs. We also keep track of the possible states,
    terminal states, and actions for each state. [Page 646]"""

    def __init__(self, init, actlist, terminals, transitions=None, reward=None, states=None, gamma=0.9):
        if not (0 < gamma <= 1):
            raise ValueError("An MDP must have 0 < gamma <= 1")

        # collect states from transitions table if not passed.
        self.states = states or self.get_states_from_transitions(transitions)

        self.init = init

        if isinstance(actlist, list):
            # if actlist is a list, all states have the same actions
      

O método **_ _init_ _** recebe os seguintes parâmetros:

- init: o estado inicial.
- actlist: Lista de ações possíveis em cada estado.
- terminais: Lista de estados terminais em que a única ação possível é sair
- gamma: Fator de desconto. Isto garante que as recompensas diferidas têm menos valor em comparação com as imediatas.

O método **R** devolve a recompensa para cada estado utilizando o dicionário self.reward.

O método **T** não está implementado. Aqui retornamos pares (probabilidade, s') onde s' pertence à lista de estados possíveis ao realizar a acção a no estado s.

O método **actions** devolve uma lista de ações possíveis em cada estado. Por predefinição, devolve todas as ações para estados diferentes dos terminais.


Vamos agora implementar o MDP simples na imagem abaixo. Os estados A e B têm as ações X e Y disponíveis. As suas probabilidades são mostradas logo acima das setas. Começamos por utilizar o MDP como classe base para o nosso *CustomMDP*. Obviamente que precisamos de fazer algumas alterações para nos adequarmos ao nosso caso. Utilizamos uma matriz de transição porque as nossas transições não são muito simples.


<img src="images/mdp-a.png">

In [8]:
# Matriz de transição num dicionário aninhado Estado -> Ações no estado -> Lista de tuplos (Probabilidade, Estado)
t = {
    "A": {
            "X": [(0.3, "A"), (0.7, "B")],
            "Y": [(1.0, "A")]
         },
    "B": {
            "X": {(0.8, "End"), (0.2, "B")},
            "Y": {(1.0, "A")}
         },
    "End": {}
}

init = "A"

terminals = ["End"]

rewards = {
    "A": 5,
    "B": -10,
    "End": 100
}

In [9]:
class CustomMDP(MDP):
    def __init__(self, init, terminals, transition_matrix, reward = None, gamma=1.0):
        # All possible actions.
        actlist = []
        for state in transition_matrix.keys():
            actlist.extend(transition_matrix[state])
        actlist = list(set(actlist))
        MDP.__init__(self, init, actlist, terminals, transition_matrix, reward, gamma=gamma)

    def T(self, state, action):
        if action is None:
            return [(0.0, state)]
        else: 
            return self.t[state][action]

Por fim, instanciamos a classe com os parâmetros do nosso MDP na imagem.

In [10]:
our_mdp = CustomMDP(init, terminals, t, rewards, gamma=0.9)

Com isto representamos com sucesso o nosso MDP. Mais tarde veremos formas de resolver este MDP.

## GRID MDP

Agora, olhamos para uma implementação concreta que faz uso do MDP como classe base.

A classe GridMDP no módulo mdp é utilizada para representar um MDP de mundo de grelha como o que é apresentado na figura seguinte:

<img src="images/grid_mdp.png">

Assumimos por agora que o ambiente é _totalmente observável_, pelo que o agente sabe sempre onde está. O código deve ser fácil de compreender se tiver lido o exemplo do CustomMDP.

In [11]:
%psource GridMDP

class GridMDP(MDP):
    """A two-dimensional grid MDP, as in [Figure 17.1]. All you have to do is
    specify the grid as a list of lists of rewards; use None for an obstacle
    (unreachable state). Also, you should specify the terminal states.
    An action is an (x, y) unit vector; e.g. (1, 0) means move east."""

    def __init__(self, grid, terminals, init=(0, 0), gamma=1.0):
        grid.reverse()  # because we want row 0 on bottom, not on top
        reward = {}
        states = set()
        self.rows = len(grid)
        self.cols = len(grid[0])
        self.grid = grid
        for x in range(self.cols):
            for y in range(self.rows):
                if grid[y][x]:
                    states.add((x, y))
                    reward[(x, y)] = grid[y][x]
        self.states = states
        actlist = orientations
        transitions = {}
        for s in states:
            transitions[s] = {}
            for a in actlist:
                transitions[s][a] = self.calculate_

O método **_ _init_ _** utiliza **grid** como parâmetro extra em comparação com a classe MDP. A grelha é uma lista aninhada de recompensas em estados.

O método **go** devolve o estado a ir numa direção específica usando vector_add.

O método **T** não está implementado e é um pouco diferente do texto. Aqui retornamos pares (probabilidade, s') onde s' pertence à lista de estados possíveis ao realizar a acção a no estado s.

O método **actions** devolve uma lista de ações possíveis em cada estado. Por predefinição, devolve todas as ações para estados diferentes dos terminais.

**to_arrows** são utilizados para representar a política num formato de grelha.

Podemos criar um GridMDP como o da figura de cima da seguinte forma:

     GridMDP([[-0.04, -0.04, -0.04, +1],
            [-0.04, None,  -0.04, -1],
            [-0.04, -0.04, -0.04, -0.04]],
            terminals=[(3, 2), (3, 1)])
 
 
 O objecto **sequential_decision_environment** no módulo mdp foi instanciado utilizando exatamente o mesmo código.

In [12]:
sequential_decision_environment

# ITERAÇÃO DE VALOR

Agora que já vimos como representar MDPs. Vamos tentar resolvê-los. O nosso objetivo final é obter uma política ideal.

Começamos por analisar a **Iteração de Valor** e uma visualização que nos deve ajudar a compreendê-la melhor.

Começamos por calcular Valor/Utilidade para cada um dos estados.

O valor de cada estado é a soma esperada das recompensas futuras descontadas, dado que começamos nesse estado e seguimos uma política específica $\pi$.

O valor ou a utilidade de um estado é dado por

$$U(s)=R(s)+\gamma\max_{a\epsilon A(s)}\sum_{s'} P(s'\ |\ s,a)U(s')$$

Esta equação chama-se equação de Bellman.

O algoritmo Iteração de Valor baseia-se na procura de soluções para esta Equação e pode ser consultado na ce´lula seguinte.

A intuição de que a **Iteração de Valor** funciona é porque os valores se propagam pelo espaço de estados através de atualizações locais. Este ponto ficará mais claro depois de encontrarmos a visualização.


In [13]:
%psource value_iteration

def value_iteration(mdp, epsilon=0.001):
    """Solving an MDP by value iteration. [Figure 17.4]"""

    U1 = {s: 0 for s in mdp.states}
    R, T, gamma = mdp.R, mdp.T, mdp.gamma
    while True:
        U = U1.copy()
        delta = 0
        for s in mdp.states:
            U1[s] = R(s) + gamma * max(sum(p * U[s1] for (p, s1) in T(s, a))
                                       for a in mdp.actions(s))
            delta = max(delta, abs(U1[s] - U[s]))
        if delta <= epsilon * (1 - gamma) / gamma:
            return U


Recebe como entradas dois parâmetros, um MDP para resolver e $\epsilon$, o erro máximo permitido na utilidade de qualquer estado. Devolve um dicionário contendo utilidades em que as chaves são os estados e os valores representam as utilidades.

A **Iteração de Valor** começa com valores iniciais arbitrários para as utilidades, calcula o lado direito da equação de Bellman e insere-o no lado esquerdo, actualizando assim a utilidade de cada estado a partir das utilidades dos seus vizinhos.

Isto é repetido até que o equilíbrio seja atingido.

Funciona com base no princípio da _Programação Dinâmica_ - utilizando informação pré-computada para simplificar a computação subsequente.

Se $U_i(s)$ for o valor de utilidade para o estado $s$ na $i$ ésima iteração, o passo da iteração, denominado actualização de Bellman, tem este aspecto:

$$ U_{i+1}(s) \leftarrow R(s) + \gamma \max_{a \epsilon A(s)} \sum_{s'} P(s'\ |\ s,a)U_{i}(s') $$

Como já deve ter reparado, `value _iteration` tem um ciclo infinito. Como decidimos quando parar de iterar?

O conceito de _contração_ explica com sucesso a convergência da iteração de valor.

No algoritmo, calculamos um valor $\delta$ que mede a diferença das utilidades do passo de tempo atual e do passo de tempo anterior:

$$\delta = \max{(\delta, \begin{vmatrix}U_{i + 1}(s) - U_i(s)\end{vmatrix})}$$

Este valor de delta diminui à medida que os valores de $U_i$ convergem.

Terminamos o algoritmo se o valor $\delta$ for inferior a um valor limite determinado pelo hiperparâmetro _epsilon_:

$$\delta \lt \epsilon \frac{(1 - \gamma)}{\gamma}$$

Para resumir, a actualização de Bellman é uma _contracção_ por um factor de $\gamma$ no espaço dos vectores de utilidade.

Segue das propriedades das contracções que em geral `value_iteration` converge sempre para uma solução única das equações de Bellman sempre que $\gamma$ é menor que 1. De seguida, terminamos o algoritmo quando se atinge uma aproximação razoável.

Na prática, acontece frequentemente que a política $\pi$ se torna óptima muito antes da função de utilidade convergir.

Para o ambiente 4 x 3 fornecido com $\gamma = 0,9$, a política $\pi$ é óptima quando $i = 4$ (na 4ª iteração), embora o erro máximo na função de utilidade seja ainda de 0,46.

Assim sendo, para aumentar a eficiência computacional, utilizamos geralmente outro método para resolver MDPs chamado **Iteração de Políticas**, que veremos a seguir.

Para já, vamos resolver o **sequential_decision_environment** GridMDP usando o `value_iteration`.

In [14]:
value_iteration(sequential_decision_environment)

{(0, 1): 0.7615582191780823,
 (1, 2): 0.8678082191780823,
 (2, 1): 0.6602739726027398,
 (0, 0): 0.7053082191780823,
 (3, 1): -1.0,
 (2, 0): 0.6114155251141552,
 (3, 0): 0.38792491121258244,
 (0, 2): 0.8115582191780822,
 (2, 2): 0.9178082191780822,
 (1, 0): 0.6553082191780822,
 (3, 2): 1.0}

The pseudocode for the algorithm:

In [15]:
pseudocode("Value-Iteration")

### AIMA3e
__function__ VALUE-ITERATION(_mdp_, _&epsi;_) __returns__ a utility function  
&emsp;__inputs__: _mdp_, an MDP with states _S_, actions _A_(_s_), transition model _P_(_s&prime;_ &vert; _s_, _a_),  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;rewards _R_(_s_), discount _&gamma;_  
&emsp;&emsp;&emsp;_&epsi;_, the maximum error allowed in the utility of any state  
&emsp;__local variables__: _U_, _U&prime;_, vectors of utilities for states in _S_, initially zero  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&delta;_, the maximum change in the utility of any state in an iteration  

&emsp;__repeat__  
&emsp;&emsp;&emsp;_U_ &larr; _U&prime;_; _&delta;_ &larr; 0  
&emsp;&emsp;&emsp;__for each__ state _s_ in _S_ __do__  
&emsp;&emsp;&emsp;&emsp;&emsp;_U&prime;_\[_s_\] &larr; _R_(_s_) &plus; _&gamma;_ max<sub>_a_ &isin; _A_(_s_)</sub> &Sigma; _P_(_s&prime;_ &vert; _s_, _a_) _U_\[_s&prime;_\]  
&emsp;&emsp;&emsp;&emsp;&emsp;__if__ &vert; _U&prime;_\[_s_\] &minus; _U_\[_s_\]  &vert; &gt; _&delta;_ __then__ _&delta;_ &larr; &vert; _U&prime;_\[_s_\] &minus; _U_\[_s_\]  &vert;  
&emsp;__until__ _&delta;_ &lt; _&epsi;_(1 &minus; _&gamma;_)&sol;_&gamma;_  
&emsp;__return__ _U_  

---
__Figure ??__ The value iteration algorithm for calculating utilities of states. The termination condition is from Equation (__??__).

---

## AIMA4e  
__function__ VALUE-ITERATION(_mdp_, _&epsi;_) __returns__ a utility function  
&emsp;__inputs__: _mdp_, an MDP with states _S_, actions _A_(_s_), transition model _P_(_s&prime;_ &vert; _s_, _a_),  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;rewards _R_(_s_,_a_,_s&prime;_), discount _&gamma;_  
&emsp;&emsp;&emsp;_&epsi;_, the maximum error allowed in the utility of any state  
&emsp;__local variables__: _U_, _U&prime;_, vectors of utilities for states in _S_, initially zero  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&delta;_, the maximum change in the utility of any state in an iteration  

&emsp;__repeat__  
&emsp;&emsp;&emsp;_U_ &larr; _U&prime;_; _&delta;_ &larr; 0  
&emsp;&emsp;&emsp;__for each__ state _s_ in _S_ __do__  
&emsp;&emsp;&emsp;&emsp;&emsp;_U&prime;_\[_s_\] &larr; max<sub>_a_ &isin; _A_(_s_)</sub> Q-VALUE(_mdp_,_s_,_a_,_U_)  
&emsp;&emsp;&emsp;&emsp;&emsp;__if__ &vert; _U&prime;_\[_s_\] &minus; _U_\[_s_\]  &vert; &gt; _&delta;_ __then__ _&delta;_ &larr; &vert; _U&prime;_\[_s_\] &minus; _U_\[_s_\]  &vert;  
&emsp;__until__ _&delta;_ &lt; _&epsi;_(1 &minus; _&gamma;_)&sol;_&gamma;_  
&emsp;__return__ _U_  

---
__Figure ??__ The value iteration algorithm for calculating utilities of states. The termination condition is from Equation (__??__).
~

## VISUALIZAÇÃO DE ITERAÇÃO DE VALOR

Para ilustrar que os valores se propagam para fora dos estados, vamos criar uma visualização simples. Utilizaremos uma versão modificada da função value_iteration que armazenará U ao longo do tempo. Também removeremos o parâmetro $\epsilon$ e, em vez disso, adicionaremos o número de iterações que desejamos.

In [16]:
def value_iteration_instru(mdp, iterations=20):
    U_over_time = []
    U1 = {s: 0 for s in mdp.states}
    R, T, gamma = mdp.R, mdp.T, mdp.gamma
    for _ in range(iterations):
        U = U1.copy()
        for s in mdp.states:
            U1[s] = R(s) + gamma * max([sum([p * U[s1] for (p, s1) in T(s, a)])
                                        for a in mdp.actions(s)])
        U_over_time.append(U)
    return U_over_time

Next, we define a function to create the visualisation from the utilities returned by **value_iteration_instru**. The reader need not concern himself with the code that immediately follows as it is the usage of Matplotib with IPython Widgets. If you are interested in reading more about these visit [ipywidgets.readthedocs.io](http://ipywidgets.readthedocs.io)

In [17]:
columns = 4
rows = 3
U_over_time = value_iteration_instru(sequential_decision_environment)

In [18]:
%matplotlib inline
from notebook import make_plot_grid_step_function

plot_grid_step = make_plot_grid_step_function(columns, rows, U_over_time)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from notebook import make_visualize

iteration_slider = widgets.IntSlider(min=1, max=15, step=1, value=0, description = "Iteração")
w=widgets.interactive(plot_grid_step,iteration=iteration_slider)
display(w)

visualize_callback = make_visualize(iteration_slider)

visualize_button = widgets.ToggleButton(description = "Visualizar", value = False)
time_select = widgets.ToggleButtons(description='Atraso extra:',options=['0', '0.1', '0.2', '0.5', '0.7', '1.0'])
a = widgets.interactive(visualize_callback, visualize = visualize_button, time_step=time_select)
display(a)

Mova o controlo deslizante acima para observar como a utilidade de cada estado muda entre iterações.

Também é possível mover o controlo deslizante utilizando as teclas de seta ou saltar para o valor editando diretamente o número com um duplo clique.

O **Botão Visualizar** animará automaticamente o controlo deslizante por si.

A **Caixa de Atraso Extra** permite definir o atraso de tempo em segundos até um segundo para cada intervalo de tempo.

Há também um editor interactivo para problemas do mundo em grelha `grid_mdp.py` na pasta gui para poder brincar.

# ITERAÇÃO DE POLÍTICA

Já vimos que a iteração de valor converge para a política ótima muito antes de se estimar com precisão a função de utilidade.

Se uma acção for claramente melhor do que todas as outras, então a magnitude exacta da utilidade nos estados envolvidos não necessita de ser precisa.

O algoritmo de iteração de políticas funciona com base nesta intuição.

O algoritmo executa dois passos fundamentais: 

* **Avaliação de políticas**: Dada uma política _&#960;&#7522;_, calcular _U&#7522; = U(&#960;&#7522;)_, a utilidade de cada estado se _&#960;&#7522;_ fosse executado.

* **Melhoria da política**: Calcule uma nova política _&#960;&#7522;&#8330;&#8321;_ utilizando uma previsão de um passo com base nos valores de utilidade já calculados.

O algoritmo termina quando a etapa de melhoria da política não produz qualquer alteração nas utilidades calculadas.

Temos agora uma versão simplificada da equação de Bellman

$$U_i(s) = R(s) + \gamma \sum_{s'}P(s'\ |\ s, \pi_i(s))U_i(s')$$

Uma observação importante nesta equação é que não tem o operador `max`, o que a torna linear.

Para _n_ estados, temos _n_ equações lineares com _n_ incógnitas, que podem ser resolvidas exactamente num tempo _**O(n&#179;)**_.

Vejamos agora como se encontra a utilidade esperada e como se implementa `policy_iteration`.

In [ ]:
%psource expected_utility

In [ ]:
%psource policy_iteration

Felizmente, não é necessário fazer uma avaliação _exacta_ da política.

Em vez disso, as utilidades podem ser razoavelmente aproximadas executando alguns passos simplificados de iteração de valor.

A equação de actualização de Bellman simplificada para o processo é :

$$U_{i+1}(s) \leftarrow R(s) + \gamma\sum_{s'}P(s'\ |\ s,\pi_i(s))U_{i}(s')$$ 

e este passo é repetido _k_ vezes para produzir a próxima estimativa de utilidade.

Este método é chamado de _iteração de política modificada_.

In [ ]:
%psource policy_evaluation

Vamos então resolver agora o **`sequential_decision_environment`** usando a `policy_iteration`.

In [ ]:
policy_iteration(sequential_decision_environment)

In [ ]:
pseudocode('Policy-Iteration')

### AIMA3e
__function__ POLICY-ITERATION(_mdp_) __returns__ a policy  
&emsp;__inputs__: _mdp_, an MDP with states _S_, actions _A_(_s_), transition model _P_(_s&prime;_ &vert; _s_, _a_)  
&emsp;__local variables__: _U_, a vector of utilities for states in _S_, initially zero  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&pi;_, a policy vector indexed by state, initially random  

&emsp;__repeat__  
&emsp;&emsp;&emsp;_U_ &larr; POLICY\-EVALUATION(_&pi;_, _U_, _mdp_)  
&emsp;&emsp;&emsp;_unchanged?_ &larr; true  
&emsp;&emsp;&emsp;__for each__ state _s_ __in__ _S_ __do__  
&emsp;&emsp;&emsp;&emsp;&emsp;__if__ max<sub>_a_ &isin; _A_(_s_)</sub> &Sigma;<sub>_s&prime;_</sub> _P_(_s&prime;_ &vert; _s_, _a_) _U_\[_s&prime;_\] &gt; &Sigma;<sub>_s&prime;_</sub> _P_(_s&prime;_ &vert; _s_, _&pi;_\[_s_\]) _U_\[_s&prime;_\] __then do__  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&pi;_\[_s_\] &larr; argmax<sub>_a_ &isin; _A_(_s_)</sub> &Sigma;<sub>_s&prime;_</sub> _P_(_s&prime;_ &vert; _s_, _a_) _U_\[_s&prime;_\]  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_unchanged?_ &larr; false  
&emsp;__until__ _unchanged?_  
&emsp;__return__ _&pi;_  

---
__Figure ??__ The policy iteration algorithm for calculating an optimal policy.

---
  
## AIMA4e  
__function__ POLICY-ITERATION(_mdp_) __returns__ a policy  
&emsp;__inputs__: _mdp_, an MDP with states _S_, actions _A_(_s_), transition model _P_(_s&prime;_ &vert; _s_, _a_)  
&emsp;__local variables__: _U_, a vector of utilities for states in _S_, initially zero  
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&pi;_, a policy vector indexed by state, initially random  
  
&emsp;__repeat__  
&emsp;&emsp;&emsp;_U_ &larr; POLICY\-EVALUATION(_&pi;_, _U_, _mdp_)  
&emsp;&emsp;&emsp;_unchanged?_ &larr; true  
&emsp;&emsp;&emsp;__for each__ state _s_ __in__ _S_ __do__  
&emsp;&emsp;&emsp;&emsp;&emsp;_a <sup> &#x2a; </sup>_ &larr; argmax<sub>_a_ &isin; _A_(_s_)</sub> Q-VALUE(_mdp_,_s_,_a_,_U_)  
&emsp;&emsp;&emsp;&emsp;&emsp;__if__ Q-VALUE(_mdp_,_s_,_a<sup>&#x2a;</sup>_,_U_) &gt; Q-VALUE(_mdp_,_s_,_&pi;_\[_s_\],_U_) __then do__    
&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;_&pi;_\[_s_\] &larr; _a<sup>&#x2a;</sup>_ ; _unchanged?_ &larr; false  
&emsp;__until__ _unchanged?_  
&emsp;__return__ _&pi;_  

---
__Figure ??__ The policy iteration algorithm for calculating an optimal policy.

## Problemas de Decisão Sequencial

Agora que já temos as ferramentas necessárias para resolver MDPs, vamos ver como os Problemas de Decisão Sequencial podem ser resolvidos passo a passo e como algumas ferramentas integradas na classe GridMDP nos ajudam a analisar melhor o problema em questão.

Como sempre, trabalharemos com o seguinte mundo da grelha:

<img src="images/grid_mdp.jpg">


Assumimos por agora que o ambiente é _totalmente observável_, pelo que o agente sabe sempre onde está.

Assumimos também que as transições são **Markovianas**, ou seja, a probabilidade de atingir o estado $s'$ a partir do estado $s$ depende apenas de $s$ e não do histórico dos estados anteriores.

Quase todos os problemas de decisão estocástica podem ser reformulados como um Processo de Decisão de Markov apenas ajustando a definição de um _estado_ para esse problema específico.

As ações do nosso agente neste ambiente, no entanto, não são fiáveis. Por outras palavras, o movimento do nosso agente é estocástico.

Mais concretamente, o agente pode:

* mover-se correctamente na direcção pretendida com uma probabilidade de _0,8_,

* mover $90^\circ$ para a direita da direcção pretendida com uma probabilidade de 0,1

* mover $90^\circ$ para a esquerda da direcção pretendida com uma probabilidade de 0,1

O agente permanece imóvel se embater numa parede.

<img src="images/grid_mdp_agent.jpg">

Estas propriedades do agente são chamadas de propriedades de transição e estão codificadas na classe GridMDP, como pode ver abaixo.

In [ ]:
%psource GridMDP.T

Para definir completamente o nosso ambiente de tarefas, precisamos de especificar a função de utilidade para o agente.

Esta é a função que dá ao agente uma estimativa aproximada de quão bom é estar num estado específico, ou quanta _recompensa_ um agente recebe por estar nesse estado.

O agente tenta então maximizar a recompensa que recebe.

Como o problema de decisão é sequencial, a função de utilidade dependerá de uma sequência de estados e não de um único estado.

Por agora, estipulamos simplesmente que em cada estado $s$, o agente recebe uma recompensa finita $R(s)$.

Para qualquer estado dado, as acções que o agente pode tomar são codificadas como abaixo:

- Mover para cima: (0, 1)

- Mover para baixo: (0, -1)

- Mover para a esquerda: (-1, 0)

- Mover para a direita: (1, 0)

- Não fazer nada: `None`

Perguntamo-nos agora como seria uma solução válida para o problema.

Não podemos ter sequências de ações fixas, pois o ambiente é estocástico e podemos acabar num estado indesejável.

Assim, uma solução deve especificar o que o agente deve fazer para _qualquer_ estado que o agente possa atingir.

Tal solução é conhecida como uma **política** e é normalmente denotada por $\pi$.

A **política óptima** é a política que produz a maior utilidade esperada e é geralmente denotada por $\pi^*$.

A classe `GridMDP` possui um método útil `to_arrows` que gera uma grelha mostrando a direcção em que o agente se deve mover, dada uma política.

Usaremos isso mais tarde para compreender melhor as propriedades do ambiente.

In [ ]:
%psource GridMDP.to_arrows

Este método codifica diretamente as ações que o agente pode tomar (descritas acima) em caracteres que representam setas e apresenta-as num formato de grelha para fins de visualização humana.

Converte a política recebida de um `dicionário` para uma grelha utilizando o método `to_grid`.

In [ ]:
%psource GridMDP.to_grid

Agora que temos todas as ferramentas necessárias e um bom entendimento do agente e do ambiente, consideramos alguns casos e vemos como o agente se deve comportar em cada caso.

### Caso 1

---

R(s) = -0,04 em todos os estados, excepto nos estados terminais

In [ ]:
# Note that this environment is also initialized in mdp.py by default
sequential_decision_environment = GridMDP([[-0.04, -0.04, -0.04, +1],
                                           [-0.04, None, -0.04, -1],
                                           [-0.04, -0.04, -0.04, -0.04]],
                                          terminals=[(3, 2), (3, 1)])

Vamos utilizar a função `best_policy` para encontrar a melhor política para este ambiente.

Mas, como pode ver, `best_policy` também requer uma função de utilidade.

Já sabemos que a função de utilidade pode ser encontrada por `value_iteration`.

Assim, a nossa melhor política é:

In [ ]:
pi = best_policy(sequential_decision_environment, value_iteration(sequential_decision_environment, .001))

Podemos agora usar o método `to_arrows` para ver como o nosso agente deve escolher as suas ações no ambiente.

In [ ]:
from utils import print_table
print_table(sequential_decision_environment.to_arrows(pi))

Esta é exactamente a saída que esperávamos:

<img src="images/-0.04.jpg">
 
Note-se que, como o custo de dar um passo é bastante pequeno em comparação com a penalização de acabar em `(4, 2)` por acidente, a política óptima é conservadora.

No estado `(3, 1)` é recomendado tomar o caminho mais longo, em vez de tomar o caminho mais curto e arriscar obter uma grande recompensa negativa de -1 em `(4, 2)`.

### Caso 2
---
R(s) = -0.4 em todos os estados menos nos estados terminais

In [ ]:
sequential_decision_environment = GridMDP([[-0.4, -0.4, -0.4, +1],
                                           [-0.4, None, -0.4, -1],
                                           [-0.4, -0.4, -0.4, -0.4]],
                                          terminals=[(3, 2), (3, 1)])

In [ ]:
pi = best_policy(sequential_decision_environment, value_iteration(sequential_decision_environment, .001))
from utils import print_table
print_table(sequential_decision_environment.to_arrows(pi))

É precisamente a configuração esperada:

<img src="images/-0.4.jpg">

Como a recompensa para cada estado é agora mais negativa, a vida é certamente mais desagradável.

O agente segue o caminho mais curto para o estado +1 e está disposto a correr o risco de cair no estado -1 por acidente.

### Caso 3
---
R(s) = -4 em todos os estados menos nos estados terminais

In [ ]:
sequential_decision_environment = GridMDP([[-4, -4, -4, +1],
                                           [-4, None, -4, -1],
                                           [-4, -4, -4, -4]],
                                          terminals=[(3, 2), (3, 1)])

In [ ]:
pi = best_policy(sequential_decision_environment, value_iteration(sequential_decision_environment, .001))
from utils import print_table
print_table(sequential_decision_environment.to_arrows(pi))

É a configuração esperada:

<ims src="images/-4.jpg">

A recompensa vitalícia para cada estado é agora inferior à do terminal menos recompensador.

A vida é tão _dolorosa_ que o agente se dirige para a saída mais próxima, pois até a pior saída é menos dolorosa do que qualquer estado de vida.

### Caso 4
---
R(s) = 4 em todos os estados menos nos estados terminais

In [ ]:
sequential_decision_environment = GridMDP([[4, 4, 4, +1],
                                           [4, None, 4, -1],
                                           [4, 4, 4, 4]],
                                          terminals=[(3, 2), (3, 1)])

In [ ]:
pi = best_policy(sequential_decision_environment, value_iteration(sequential_decision_environment, .001))
from utils import print_table
print_table(sequential_decision_environment.to_arrows(pi))

A configuração esperada é a seguinte:

<img src="images/4.jpg">

Como a vida é positivamente agradável e o agente evita _ambas_ as saídas.

Mesmo que o resultado obtido não seja exatamente o que pretendemos, não está definitivamente errado.

O cenário aqui exige que o agente faça tudo menos atingir um estado terminal, pois esta é a única forma de o agente maximizar a sua recompensa (a recompensa total tende para o infinito), e o programa faz exatamente isso.

Atualmente, a classe GridMDP não suporta um marcador explícito para uma ação "faça o que quiser" ou uma condição "não se importe". Experimente você introduzir essa ação.

---
## Apêndice

Surpreendentemente, verifica-se que existem seis outras políticas ótimas para vários intervalos de R(s).

Pode tentar descobrir por si mesmo. Para o ajudar nessa tarefa temos um editor GridMDP em `grid_mdp.py`.

Aqui está um breve tutorial sobre como usá-lo

Vamos usá-lo para resolver o `Caso 2` acima

1.º Execute `python grid_mdp.py` a partir da pasta corrente.

2.º Introduza as dimensões da grelha (3 x 4 neste caso) e clique em `'Build a GridMDP'`

3.º Clique em `Initialize` no menu `Edit`.

4.º Defina a recompensa para -0,4 e clique em `Aplicar`. Sair do diálogo.

<img src="images/ge0.jpg">

5.º Seleccione a célula (1, 1) e marque o botão de opção `Wall`. `Aplicar` e sair do diálogo.

<img src="images/ge1.jpg">

6.º Seleccione as células (4, 1) e (4, 2) e marque o botão de opção `Terminal` para ambas. Defina as recompensas adequadamente e clique em `Aplicar`. Sair do diálogo. A sua janela deve ficar mais ou menos assim.

<img src="images/ge2.jpg">

7.º Está pronto agora. Clique em `Construir e Executar` no menu `Construir` e observe o mapa de calor a calcular a função de utilidade.

<img src="images/ge4.jpg">

Os tons verdes indicam utilidades positivas e os tons castanhos indicam utilidades negativas.

Os valores da função de utilidade e do diagrama de setas aparecerão em caixas de diálogo separadas após a convergência do algoritmo.